In [0]:
import os
import sys
from databricks.sdk.runtime import dbutils

cod_pipelineID = dbutils.widgets.get("cod_pipelineID")
input_params = dbutils.widgets.get("input_params")
RunId = dbutils.widgets.get("RunId")
snapshot_date = dbutils.widgets.get("snapshot_date")
cod_layerName = dbutils.widgets.get("cod_layerName")
cod_productName = dbutils.widgets.get("cod_productName")
env = dbutils.widgets.get("env")
flg_lastSnap = 1

sys_path = f'/Workspace/AI/{cod_productName}/{cod_productName}_model/files'
notebook_path = "/Workspace/" + os.path.dirname(
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
sys.path.append("..")
os.chdir(notebook_path)

In [0]:
print(f"cod_pipelineID -> {cod_pipelineID}")
print(f"input_params -> {input_params}")
print(f"RunId -> {RunId}")
print(f"snapshot_date -> {snapshot_date}")
print(f"cod_layerName -> {cod_layerName}")
print(f"cod_productName -> {cod_productName}")
print(f"env -> {env}")
print(f"flg_lastSnap -> {flg_lastSnap}")

In [0]:
import pyspark.sql.functions as F
from datetime import datetime
import pandas as pd
from pandas import DataFrame as pandasDataFrame
from pyspark.sql import DataFrame
from dateutil.relativedelta import relativedelta
from typing import Dict
import logging
import json
import ast
import hashlib
from src.masters.training_masters import *

In [0]:
%run ./materialization_utils/materialization_utils

In [0]:
general_utils = GeneralUtils(spark)
metric_materializer = MetricMaterializer(spark)
logger = general_utils.set_logging()
master_materilizer = masterMaterilizer(spark, logger)

config = general_utils.load_config(general_utils.get_variable(cod_productName)['config_setup_path'])
metadata_path = general_utils.get_variable(cod_productName)['metadata_path']

# snapshot_date = general_utils.date_to_start_month(snapshot_date)
catalog_name = config['environment'][env]['catalog']
schema_name = config['environment'][env]['schema']

try:
    input_params_json = ast.literal_eval(input_params)
except (ValueError, SyntaxError) as e:
    input_params_json = json.loads(input_params)

table_name = input_params_json['name']

In [0]:
if cod_productName == "creri":
    if cod_layerName == "INFER_SERVING":
        print(snapshot_date)
    else:
        snapshot_date = general_utils.date_to_start_month(snapshot_date)
        try:
            snapshot_date = general_utils.replace_month_date(snapshot_date, int(input_params_json['today_minus_n_months']))
        except:
            snapshot_date = general_utils.replace_month_date(snapshot_date, 0)
        print(snapshot_date)

In [0]:
try:
    entita_by_name = {
        entita['class_name']+"_name": entita['name']
        for entita in config['metrics']
    } | {
        entita['class_name']+"_name": entita['name']
        for entita in config['targets']
    }
    entita_by_name.update({entita['class_name']+"_keys": entita.get('join_key', []) for entita in config['metrics']})
    entita_by_name.update({entita['class_name']+"_keys": entita.get('join_key', []) for entita in config['targets']})

    cb_by_name = {
        entita['class_name']: entita['name']
        for entita in config['customer_bases']
        if entita['type'] == 'PREP_CUSTOMER_BASE'
    }

    #### if data snapshot overwrite is required, delete the exist snapshot data ####
    snapshot_overwrite_res = metric_materializer.check_snapshot_overwrite(catalog_name, 
                            schema_name, 
                            table_name, 
                            snapshot_date, 
                            input_params_json['snapshot_overwrite'])
    logger.info(f"{snapshot_overwrite_res}")

    metrics_used_list = metric_materializer.get_metrics_used(config, input_params_json)
    hash_data_version = metric_materializer.get_met_version(catalog_name, schema_name, metrics_used_list, input_params_json['class_name'], master_materilizer.get_version(), snapshot_date)

    #### check if metric is updated on the latest version ####
    try:
        logger.info(f"checking if metric is updated on the latest version...")
        is_metrics_updated = metric_materializer.is_metrics_updated(catalog_name,
                                            schema_name, 
                                            table_name, 
                                            hash_data_version, 
                                            snapshot_date)
        logger.info(f"is metrics updated on the latest version? {is_metrics_updated}")
        logger.info(f"metric updating version checked")
    except Exception as e:
        logger.info("Error metric updating:" + str(e))
        raise

    #### if metric is not updated -> select the snapshot date not. updated #### 
    if not is_metrics_updated:
        logger.info(f"=======================================================================")
        logger.info(f"UPDATING OLD SNAPSHOT DATE PHASE IS STARTING...")
        logger.info(f"identifying old snapshot date to update...")
        data_version_collection = (
            spark.read.table(f"{catalog_name}.{schema_name}.{table_name}") \
            .where(f"flg_lastSnap = 1")\
            .select("snapshot_date", "version") \
            .orderBy("snapshot_date", F.col("version").desc()) \
            .dropDuplicates(["snapshot_date"]) \
            .toLocalIterator()
        )
        logger.info(f"old snapshot date to update identified")

        #### loop on the snapshot date not updated and materialize them #### 
        for data_object in data_version_collection:
            if data_object["version"] != hash_data_version:
                logger.info(f"UPDATING snapshot_date: {data_object['snapshot_date']}")

                #### set to 0 the old data need to be updated (per snapshot date) ####
                logger.info(f"closing old records...")
                metric_versioning = metric_materializer.metric_versioning(catalog_name, schema_name, table_name, data_object["snapshot_date"], input_params_json['historicization'])
                logger.info(f"closed old records with flg_lastSnap to 0")

                #### materialize the new data on the selected snapshot date ####
                logger.info(f"materializing data in memory...")
                result_df = master_materilizer.get_dataframe_as_of(entita_by_name, cb_by_name, input_params_json, catalog_name, schema_name, table_name, data_object['snapshot_date'], metric_materializer)
                logger.info(f"data materialized")

                #### cast fields based on the configuration metadata (schema enforcement) ####
                logger.info("casting fields...")
                schema = metric_materializer.remove_tech_columns(spark.read.table(f"{catalog_name}.{schema_name}.{table_name}")).schema
                schema_map = {field.name: field.dataType for field in schema.fields}
                common_cols, df_only_cols = metric_materializer.get_schema_fields(schema_map, result_df)
                result_df_casted = metric_materializer.cast_columns(common_cols, schema_map, result_df)
                logger.info("fields casted based on configuration data")

                #### adding technical columns ####
                logger.info(f"generating tech columns...")
                result_df_final = metric_materializer.add_tech_columns(
                    result_df_casted, 
                    datetime.now().strftime("%Y-%m-%d %H:%M:%S"), 
                    RunId, 
                    hash_data_version, 
                    flg_lastSnap,
                    input_params_json['join_key'], 
                    data_object['snapshot_date'])
                logger.info(f"tech columns generated")
        
                #### write data on the target table ####
                logger.info(f"writing data to destination table...")
                general_utils.write_table(
                    catalog_name, 
                    schema_name, 
                    table_name, 
                    result_df_final, 
                    "append")
                logger.info(f"data successfully written on {catalog_name}.{schema_name}.{table_name}")
        
                #### populate technical column end_date closing the write operation ####
                logger.info(f"populating end date...")
                metric_materializer.populate_metrics_end_date(
                catalog=catalog_name, 
                database=schema_name, 
                table=table_name,
                snapshot_date=data_object['snapshot_date'])
                logger.info(f"technical column successfully populated")
                logger.info(f"snapshot_date {data_object['snapshot_date']} UPDATED")
            else:
                logger.info(f"Updating snapshot_date: Materialized data updated for snapshot_date = {snapshot_date}")
            logger.info(f"UPDATING OLD SNAPSHOT DATE PHASE SUCCESSFULLY END")
    else:
        logger.info(f"Materialized data updated!!")

    #### check if the snapshot date is already materialized ####
    if metric_materializer.check_snapshot_data(f"{catalog_name}.{schema_name}.{table_name}", snapshot_date):
        logger.info(f"=======================================================================")
        logger.info(f"Materialization with the selected snapshot date already exists")
    else:
        logger.info(f"=======================================================================")
        logger.info(f"UPDATING CURRENT SNAPSHOT DATE PHASE IS STARTING...")
        #### set to 0 the old data need to be updated (per snapshot date) ####
        logger.info(f"closing old records...")
        metric_versioning = metric_materializer.metric_versioning(catalog_name, schema_name, table_name, snapshot_date, input_params_json['historicization'])
        logger.info(f"{metric_versioning}")
        logger.info(f"closed old records with flg_lastSnap to 0")

        #### materialize the new data on the selected snapshot date ####
        logger.info(f"materializing data in memory...")
        result_df = master_materilizer.get_dataframe_as_of(entita_by_name, cb_by_name, input_params_json, catalog_name, schema_name, table_name, snapshot_date, metric_materializer)
        logger.info(f"data materialized")

        #### cast fields based on the configuration metadata (schema enforcement) ####
        logger.info("casting fields...")
        schema = metric_materializer.remove_tech_columns(spark.read.table(f"{catalog_name}.{schema_name}.{table_name}")).schema
        schema_map = {field.name: field.dataType for field in schema.fields}
        common_cols, df_only_cols = metric_materializer.get_schema_fields(schema_map, result_df)
        result_df_casted = metric_materializer.cast_columns(common_cols, schema_map, result_df)
        logger.info("fields casted based on configuration data")

        #### adding technical columns ####
        logger.info(f"generating tech columns...")
        result_df_final = metric_materializer.add_tech_columns(
            result_df_casted, 
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"), 
            RunId, 
            hash_data_version, 
            flg_lastSnap,
            input_params_json['join_key'], 
            snapshot_date)
        logger.info(f"tech columns generated")

        #### write data on the target table ####
        logger.info(f"writing data to destination table...")
        general_utils.write_table(
            catalog_name, 
            schema_name, 
            table_name, 
            result_df_final, 
            "append")
        logger.info(f"Total count written: {result_df_final.count()}")
        logger.info(f"data successfully written on {catalog_name}.{schema_name}.{table_name}")

        #### populate technical column end_date closing the write operation ####
        logger.info(f"populating end date...")
        metric_materializer.populate_metrics_end_date(
        catalog=catalog_name, 
        database=schema_name, 
        table=table_name,
        snapshot_date=snapshot_date)

        logger.info(f"technical column successfully populated")
        logger.info(f"snapshot_date {data_object['snapshot_date']} UPDATED")
except Exception as e:
    logger.info("Error on materialization:" + str(e))
    raise

In [0]:
dbutils.notebook.exit("Success!!!!")

---